In [1]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# 1 Load data

## 1.1 Load population

In [2]:
pop_df = pd.read_csv('Pop ML V PIPE.csv', encoding='latin-1', sep='|')
pop_df.head(10)

,MSISDN,STATE_IN,SUBSCRIBER_TYPE_IN,RATE_PLAN,ACCOUNT_ACTIVATED_DATE
0,21697077044,ACTIVE,PREPAID,PRE - Classic,2003-12-19
1,21697076876,ACTIVE,PREPAID,PRE - Classic,2021-11-23
2,21697072547,ACTIVE,PREPAID,PRE - Classic,2020-09-14
3,21697068526,ACTIVE,PREPAID,PRE - Classic,2018-04-11
4,21697061045,ACTIVE,PREPAID,PRE - Classic,2003-11-15
5,21697060361,ACTIVE,PREPAID,PRE - Classic,2003-11-06
6,21697057138,ACTIVE,PREPAID,PRE - Classic,2004-02-01
7,21697046515,ACTIVE,PREPAID,PRE - Classic,2023-04-22
8,21697038087,SUSPENDED,PREPAID,PRE - Classic,2024-03-10
9,21697026893,SUSPENDED,PREPAID,PRE - Classic,2025-01-25


## 1.2 Load SOS

In [3]:
sos_df = pd.read_csv('DS_SOS_ML.csv', encoding='latin-1', sep='|')
sos_df.head(10)

,MSISDN,LOAN_ID,SOS_TYPE,CREDIT_DATE,CREDIT_AMOUNT,TOTAL_REIMBURSED,OUTSTANDING_AMOUNT,LAST_REIMBURSE_DATE,REIMBURSE_RATIO,REIMBURSE_STATUS,DAYS_SINCE_CREDIT
0,21650142555,528139435,CVAS_SOS_CR_DATA,09/01/2026,"4,5","4,5",0,23/01/2026,1,FULL,90
1,21693464197,528150918,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
2,21697349073,528136747,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,10/01/2026,1,FULL,90
3,21698641675,528163694,CVAS_SOS_CR_SOLDE,09/01/2026,3,3,0,15/01/2026,1,FULL,90
4,21694195742,528176161,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,09/01/2026,1,FULL,90
5,21693574965,528132046,CVAS_SOS_CR_SOLDE,09/01/2026,"2,5","2,5",0,19/01/2026,1,FULL,90
6,21696646805,528207682,CVAS_SOS_CR_SOLDE,09/01/2026,5,5,0,19/01/2026,1,FULL,90
7,21693370117,528155180,CVAS_SOS_CR_SOLDE,09/01/2026,"0,5","0,5",0,09/01/2026,1,FULL,90
8,21697385533,528223219,CVAS_SOS_CR_DATA,09/01/2026,"0,25","0,25",0,09/01/2026,1,FULL,90
9,21695867328,528161149,CVAS_SOS_CR_DATA,09/01/2026,"0,9","0,9",0,10/01/2026,1,FULL,90


# Data interpretation 

In [4]:
pop_df.isnull().sum()

MSISDN                       0
STATE_IN                     0
SUBSCRIBER_TYPE_IN        4442
RATE_PLAN                 4442
ACCOUNT_ACTIVATED_DATE    4442
dtype: int64

In [5]:
sos_df.isnull().sum()

MSISDN                       0
LOAN_ID                      0
SOS_TYPE                     0
CREDIT_DATE                  0
CREDIT_AMOUNT                0
TOTAL_REIMBURSED             0
OUTSTANDING_AMOUNT           0
LAST_REIMBURSE_DATE    2946503
REIMBURSE_RATIO              0
REIMBURSE_STATUS             0
DAYS_SINCE_CREDIT            0
dtype: int64

## Possible  Columns values for population file

In [6]:
for col in pop_df.columns:
    print(f"{col}:")
    print(pop_df[col].unique())
    print("-" * 40)

MSISDN:
[21697077044 21697076876 21697072547 ... 21699004086 21655565721
 21698400893]
----------------------------------------
STATE_IN:
['ACTIVE' 'SUSPENDED' 'ON-HOLD' 'Disconnected Network']
----------------------------------------
SUBSCRIBER_TYPE_IN:
['PREPAID' 'HYB' 'POSTPAID' nan]
----------------------------------------
RATE_PLAN:
['PRE - Classic' 'PRE - Ola' 'PRE - LOL' 'MOBI Plafonné'
 'Corporate Optimum Plus' 'PRE - Club Optimum Plus'
 'PRE - Corporate Optimum' 'PRE - Corporate Optimum Family' 'PRE - Tawwa'
 'Forfait SNJT' 'PRE - Employe TT' 'PRE - TAWWA REINSTALLED'
 'PRE - Double Street' 'PRE - Messenger' 'PRE - Corporate Optimum TT'
 'Forfaits Mobile TT' 'Forfaits Mobile PRO' 'PRE - Trankil TT' 'OPTICA'
 'PRE - 900 bonus' 'PRE - offre 40' 'PRE - Classe_Regions'
 'PRE - Oulidha 1000%' 'PRE - TT 1000%' 'PRE - Day Pass' 'Hayya'
 'Forfait SELECT' 'PRE - Double Reinstal' 'PRE - AHLA' 'PRE - Doublé'
 'PRE - Binetna' 'PRE - Conso 500mil_TT' 'PRE - TT 300%'
 'PRE - E.M. 1000%' 'PR

## Possible Columns values for SOS file

In [7]:
for col in sos_df.columns:
    print(f"{col}:")
    print(sos_df[col].unique())
    print("-" * 40)

MSISDN:
[21650142555 21693464197 21697349073 ... 21697580810 21699605391
 21690563069]
----------------------------------------
LOAN_ID:


[528139435 528150918 528136747 ... 541128113 545729225 544669242]
----------------------------------------
SOS_TYPE:
['CVAS_SOS_CR_DATA' 'CVAS_SOS_CR_SOLDE']
----------------------------------------
CREDIT_DATE:
['09/01/2026' '10/01/2026' '11/01/2026' '12/01/2026' '13/01/2026'
 '14/01/2026' '15/01/2026' '16/01/2026' '17/01/2026' '18/01/2026'
 '19/01/2026' '20/01/2026' '21/01/2026' '22/01/2026' '23/01/2026'
 '24/01/2026' '25/01/2026' '26/01/2026' '27/01/2026' '28/01/2026'
 '29/01/2026' '30/01/2026' '31/01/2026' '01/02/2026' '02/02/2026'
 '03/02/2026' '04/02/2026' '05/02/2026' '06/02/2026' '07/02/2026'
 '08/02/2026' '09/02/2026' '10/02/2026' '11/02/2026' '12/02/2026'
 '13/02/2026' '14/02/2026' '15/02/2026' '16/02/2026' '17/02/2026'
 '18/02/2026' '19/02/2026' '20/02/2026' '21/02/2026' '22/02/2026'
 '23/02/2026' '24/02/2026' '25/02/2026' '26/02/2026' '27/02/2026'
 '28/02/2026' '01/03/2026' '02/03/2026' '03/03/2026' '04/03/2026'
 '05/03/2026' '06/03/2026' '07/03/2026' '08/03/2026' '09/03/20

In [8]:
# Data Preparation for Model

In [9]:
# Merge datasets on MSISDN
df = sos_df.merge(pop_df[['MSISDN', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']], 
                   on='MSISDN', how='inner')

print(f"Merged dataset shape: {df.shape}")
print(f"\nColumns in merged dataset: {df.columns.tolist()}")
print(f"\nNull values after merge:")
print(df[['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']].isnull().sum())

Merged dataset shape: (18562478, 14)

Columns in merged dataset: ['MSISDN', 'LOAN_ID', 'SOS_TYPE', 'CREDIT_DATE', 'CREDIT_AMOUNT', 'TOTAL_REIMBURSED', 'OUTSTANDING_AMOUNT', 'LAST_REIMBURSE_DATE', 'REIMBURSE_RATIO', 'REIMBURSE_STATUS', 'DAYS_SINCE_CREDIT', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE']

Null values after merge:
SOS_TYPE                      0
SUBSCRIBER_TYPE_IN        35004
RATE_PLAN                 35004
ACCOUNT_ACTIVATED_DATE    35004
CREDIT_DATE                   0
CREDIT_AMOUNT                 0
dtype: int64


In [10]:
# Data Cleaning - Remove rows with null values in required features
required_cols = ['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN', 'ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE', 'CREDIT_AMOUNT']
df_clean = df.dropna(subset=required_cols).copy()

print(f"Dataset shape after removing nulls: {df_clean.shape}")
print(f"Rows removed: {df.shape[0] - df_clean.shape[0]}")
print(f"\nRemaining null values:")
print(df_clean[required_cols].isnull().sum())

Dataset shape after removing nulls: (18527474, 14)
Rows removed: 35004

Remaining null values:
SOS_TYPE                  0
SUBSCRIBER_TYPE_IN        0
RATE_PLAN                 0
ACCOUNT_ACTIVATED_DATE    0
CREDIT_DATE               0
CREDIT_AMOUNT             0
dtype: int64


In [ ]:
# Convert date columns to datetime
df_clean['ACCOUNT_ACTIVATED_DATE'] = pd.to_datetime(df_clean['ACCOUNT_ACTIVATED_DATE'], errors='coerce')
df_clean['CREDIT_DATE'] = pd.to_datetime(df_clean['CREDIT_DATE'], errors='coerce')

# Remove rows where date conversion failed
df_clean = df_clean.dropna(subset=['ACCOUNT_ACTIVATED_DATE', 'CREDIT_DATE'])

print(f"Dataset shape after date conversion: {df_clean.shape}")
print(f"\nDate columns info:")
print(f"ACCOUNT_ACTIVATED_DATE - min: {df_clean['ACCOUNT_ACTIVATED_DATE'].min()}, max: {df_clean['ACCOUNT_ACTIVATED_DATE'].max()}")
print(f"CREDIT_DATE - min: {df_clean['CREDIT_DATE'].min()}, max: {df_clean['CREDIT_DATE'].max()}")

In [ ]:
# Feature Engineering - Derive new columns
# ACCOUNT_AGE_DAYS: Days between account activation and credit date
df_clean['ACCOUNT_AGE_DAYS'] = (df_clean['CREDIT_DATE'] - df_clean['ACCOUNT_ACTIVATED_DATE']).dt.days

# CREDIT_DAY_OF_WEEK: Day of week when credit was given (0=Monday, 6=Sunday)
df_clean['CREDIT_DAY_OF_WEEK'] = df_clean['CREDIT_DATE'].dt.dayofweek

# CREDIT_MONTH: Month of the credit date
df_clean['CREDIT_MONTH'] = df_clean['CREDIT_DATE'].dt.month

print("Derived features created:")
print(f"ACCOUNT_AGE_DAYS - min: {df_clean['ACCOUNT_AGE_DAYS'].min()}, max: {df_clean['ACCOUNT_AGE_DAYS'].max()}, mean: {df_clean['ACCOUNT_AGE_DAYS'].mean():.2f}")
print(f"CREDIT_DAY_OF_WEEK - unique values: {sorted(df_clean['CREDIT_DAY_OF_WEEK'].unique())}")
print(f"CREDIT_MONTH - unique values: {sorted(df_clean['CREDIT_MONTH'].unique())}")

print(f"\nData types:")
print(df_clean[['ACCOUNT_AGE_DAYS', 'CREDIT_DAY_OF_WEEK', 'CREDIT_MONTH']].dtypes)

Derived features created:
ACCOUNT_AGE_DAYS - min: -95, max: 9484, mean: 2067.40
CREDIT_DAY_OF_WEEK - unique values: [0, 1, 2, 3, 4, 5, 6]
CREDIT_MONTH - unique values: [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]

Data types:
ACCOUNT_AGE_DAYS      int64
CREDIT_DAY_OF_WEEK    int32
CREDIT_MONTH          int32
dtype: object


In [ ]:
# Step 1: Analyze cardinality and determine encoding strategy
df_model = df_clean.copy()
categorical_cols = ['SOS_TYPE', 'SUBSCRIBER_TYPE_IN', 'RATE_PLAN']
numeric_cols = ['ACCOUNT_AGE_DAYS', 'CREDIT_DAY_OF_WEEK', 'CREDIT_MONTH']

print("=" * 70)
print("CARDINALITY ANALYSIS & ENCODING STRATEGY")
print("=" * 70)

encoder_strategy = {}
for col in categorical_cols:
    n_unique = df_model[col].nunique()
    # If unique values <= 20: OneHotEncoder, else OrdinalEncoder
    encoder_type = "OneHotEncoder" if n_unique <= 20 else "OrdinalEncoder"
    encoder_strategy[col] = {
        'unique_values': n_unique,
        'encoder': encoder_type
    }
    print(f"{col}: {n_unique} unique values → {encoder_type}")

print(f"\nTarget variable (CREDIT_AMOUNT):")
print(f"  Min: {df_model['CREDIT_AMOUNT'].min():.2f}")
print(f"  Max: {df_model['CREDIT_AMOUNT'].max():.2f}")
print(f"  Mean: {df_model['CREDIT_AMOUNT'].mean():.2f}")
print(f"\nTotal records: {len(df_model)}")
print("=" * 70)

SOS_TYPE - unique values: 2, encoding: {'CVAS_SOS_CR_DATA': 0, 'CVAS_SOS_CR_SOLDE': 1}
SUBSCRIBER_TYPE_IN - unique values: 3, encoding: {'HYB': 0, 'POSTPAID': 1, 'PREPAID': 2}
RATE_PLAN - unique values: 99, encoding: {'Aziza Mobile 35Mil': 0, 'Cle 3G Prepaye': 1, 'Club Optimum UGTT': 2, 'Corporate Optimum Plus': 3, 'Dim@Connect Hyb': 4, 'Dim@net Corporate hybride': 5, 'Forfait DATA Only Prépayé': 6, 'Forfait SELECT': 7, 'Forfait SNJT': 8, 'Forfaits Mobile PRO': 9, 'Forfaits Mobile TT': 10, 'Forfaits Mobiles Plafonnés OH!': 11, 'Hayya': 12, 'MOBI Illimite Plafonné': 13, 'MOBI Plafonné': 14, 'MOBI Plus Plafonné': 15, 'MOBI+ Plafonné illimité': 16, 'NETBOX Plus Prépayée': 17, 'OPTICA': 18, 'Offre à la carte': 19, 'PRE - 1=11': 20, 'PRE - 900 bonus': 21, 'PRE - AHLA': 22, 'PRE - Ahla Bonnusset': 23, 'PRE - Binetna': 24, 'PRE - Binetna Reinstal': 25, 'PRE - CSS 1000% New': 26, 'PRE - CSS MOBILE': 27, 'PRE - CSS Mobile 1000%': 28, 'PRE - CSS Mobile 1500%%': 29, 'PRE - CSS Mobile 2000%': 30, 

: 

: 

In [ ]:
# Prepare features and target for modeling (BEFORE train/test split)
# Features: numeric + categorical (NOT yet encoded)
X = df_model[numeric_cols + categorical_cols].copy()
y = df_model['CREDIT_AMOUNT'].copy()

# Verify data integrity
print(f"Dataset size: {X.shape[0]} rows × {X.shape[1]} features")
print(f"\nNull values in features:")
print(X.isnull().sum())
print(f"\nFeature dtypes:")
print(X.dtypes)
print(f"\nNumeric features summary:")
print(X[numeric_cols].describe())

# Train-Test Split (80-20) - BEFORE fitting any transformers (prevent data leakage)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(f"\n{'='*70}")
print(f"Train set: {X_train.shape[0]} samples")
print(f"Test set: {X_test.shape[0]} samples")
print(f"Train/Test ratio: {X_train.shape[0]/X_test.shape[0]:.1f}:1")
print(f"{'='*70}")

In [ ]:
# Step 1: Build preprocessing transformers based on cardinality
categorical_transformers = []

for col in categorical_cols:
    n_unique = df_model[col].nunique()
    
    if n_unique <= 20:
        # OneHotEncoder for low cardinality (creates binary features)
        transformer = OneHotEncoder(
            categories='auto',
            sparse_output=False,
            handle_unknown='ignore',  # Handles unknown categories gracefully
            dtype='float32'
        )
    else:
        # OrdinalEncoder for high cardinality (more memory efficient)
        transformer = OrdinalEncoder(
            handle_unknown='use_encoded_value',
            unknown_value=-1,  # Assign -1 for unknown values
            dtype='float32'
        )
    
    categorical_transformers.append((col, transformer, [col]))

# Step 2: Create ColumnTransformer (fit only on training data to prevent leakage)
preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', StandardScaler(), numeric_cols),
        *categorical_transformers
    ],
    remainder='passthrough',
    verbose_feature_names_out=False
)

# Step 3: Create full ML pipeline
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LinearRegression())
])

# Fit pipeline on training data ONLY (preprocessor learns from X_train, model learns from y_train)
full_pipeline.fit(X_train, y_train)

# Make predictions
y_train_pred = full_pipeline.predict(X_train)
y_test_pred = full_pipeline.predict(X_test)

# Evaluate model
train_mse = mean_squared_error(y_train, y_train_pred)
test_mse = mean_squared_error(y_test, y_test_pred)
train_rmse = np.sqrt(train_mse)
test_rmse = np.sqrt(test_mse)
train_r2 = r2_score(y_train, y_train_pred)
test_r2 = r2_score(y_test, y_test_pred)

print("=" * 70)
print("LINEAR REGRESSION MODEL RESULTS (WITH BEST PRACTICES)")
print("=" * 70)
print(f"\n✓ Encoding Strategy Applied:")
for col, strategy in encoder_strategy.items():
    print(f"  {col}: {strategy['unique_values']} values → {strategy['encoder']}")

print(f"\n✓ Data Leakage Prevention:")
print(f"  - Preprocessor fitted on training data only")
print(f"  - StandardScaler fitted on training data only")
print(f"  - Unknown categories handled gracefully")

print(f"\nTRAINING SET METRICS:")
print(f"  MSE:  {train_mse:.4f}")
print(f"  RMSE: {train_rmse:.4f}")
print(f"  R²:   {train_r2:.4f}")

print(f"\nTEST SET METRICS:")
print(f"  MSE:  {test_mse:.4f}")
print(f"  RMSE: {test_rmse:.4f}")
print(f"  R²:   {test_r2:.4f}")

# Get model coefficients
model = full_pipeline.named_steps['model']
print(f"\nMODEL DETAILS:")
print(f"  Total features after preprocessing: {len(model.coef_)}")
print(f"  Intercept: {model.intercept_:.6f}")

print("\n" + "=" * 70)